# Qwen Endpoint Test

Run the cells from top to bottom. This notebook loads the repo-local `configs/qwen.remote.local.env`, authenticates to the OpenAI-compatible Qwen endpoint, sends `chat_template_kwargs={"enable_thinking": False}`, and normalizes either `content` or `reasoning_content` into clear visible output.


In [ ]:
# 1) Imports and repo-local env loading
import json
import os
import re
from pathlib import Path

import httpx
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists() or (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find the traj-eval repo root from the current notebook path.")


def clean_env(value):
    if value is None:
        return None
    return value.strip().strip('"').strip("'")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ENV_PATH = Path(os.getenv("TRAJ_EVAL_PROVIDER_ENV", REPO_ROOT / "configs" / "qwen.remote.local.env")).expanduser()
loaded = load_dotenv(ENV_PATH, override=True)

BASE_URL = clean_env(os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE"))
API_KEY = clean_env(os.getenv("OPENAI_API_KEY"))
MODEL = clean_env(os.getenv("CMBAGENT_EVAL_LOCAL_MODEL") or os.getenv("TRAJ_EVAL_MODEL") or os.getenv("OPENAI_MODEL"))

display({
    "cell": "1 / repo-local env loading",
    "repo_root": str(REPO_ROOT),
    "env_path": str(ENV_PATH),
    "env_exists": ENV_PATH.exists(),
    "env_loaded": loaded,
    "base_url": BASE_URL,
    "api_key_set": bool(API_KEY),
    "model_from_env": MODEL or "auto-detect from /models",
})

assert ENV_PATH.exists(), f"Env file not found: {ENV_PATH}"
assert BASE_URL, "Missing OPENAI_BASE_URL or OPENAI_API_BASE in qwen.remote.local.env"
assert API_KEY, "Missing OPENAI_API_KEY in qwen.remote.local.env"

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    timeout=180.0,
    max_retries=0,
)


In [ ]:
# 2) Authenticated /models probe and model selection
headers = {"Authorization": f"Bearer {API_KEY}"}

with httpx.Client(timeout=20.0, headers=headers) as h:
    models_response = h.get(f"{BASE_URL.rstrip('/')}/models")

models_text = models_response.text

display({
    "cell": "2 / authenticated models probe",
    "status_code": models_response.status_code,
    "body_preview": models_text[:2000],
})

models_response.raise_for_status()

models_json = models_response.json()
model_ids = [item.get("id") for item in models_json.get("data", []) if item.get("id")]

if not MODEL and model_ids:
    MODEL = model_ids[0]

display({"available_models": model_ids, "selected_model": MODEL})

assert MODEL, "The provider answered /models, but no model id was returned. Set CMBAGENT_EVAL_LOCAL_MODEL in qwen.remote.local.env."

if model_ids and MODEL not in model_ids:
    display({
        "warning": "MODEL from env was not listed by /models. Keeping the env model, but you may need to choose one of available_models.",
        "env_model": MODEL,
    })


In [ ]:
# 3) Response normalizer copied from the project strategy

def strip_template_tokens(text: str) -> str:
   cleaned = text.strip()
   if not cleaned:
       return cleaned
   cleaned = re.sub(r"<\|im_start\|>[^\\|>]*\|?>?\s*", "", cleaned, flags=re.IGNORECASE)
   cleaned = re.sub(r"<\|im_end\|>", "", cleaned, flags=re.IGNORECASE)
   cleaned = re.sub(r"\bim_start\|>|\bim_end\|>", "", cleaned, flags=re.IGNORECASE)
   cleaned = re.sub(r"</?s>", "", cleaned, flags=re.IGNORECASE)
   cleaned = re.sub(r"<\|[^|]+\|>", "", cleaned, flags=re.IGNORECASE)
   return cleaned.strip()


def extract_final_from_reasoning(text: str) -> str:
   markers = ["Final Output Generation:", "Final Answer:", "Final:", "Answer:"]
   for marker in markers:
       if marker in text:
           tail = text.split(marker)[-1].strip()
           bullet_match = re.search(r"\*\s*(.+)", tail)
           if bullet_match:
               return bullet_match.group(1).strip().strip('"')
           return tail.strip().strip('"')
   return text.strip()


def normalize_qwen_body(body: dict) -> str:
   choice = (body.get("choices") or [{}])[0]
   message = choice.get("message") or {}
   content = str(message.get("content") or "")
   reasoning = str(
       message.get("reasoning_content")
       or (message.get("provider_specific_fields") or {}).get("reasoning_content")
       or ""
   )
   text = content.strip() or extract_final_from_reasoning(reasoning)
   return strip_template_tokens(text).strip()


def normalize_qwen_response(response) -> str:
   return normalize_qwen_body(response.model_dump())


def show_qwen_body(body: dict, label: str = "qwen response") -> str:
   choice = (body.get("choices") or [{}])[0]
   message = choice.get("message") or {}
   visible_text = normalize_qwen_body(body)

   display({
       "cell": label,
       "finish_reason": choice.get("finish_reason"),
       "content_repr": repr(message.get("content")),
       "reasoning_preview": str(message.get("reasoning_content") or "")[:1200],
       "provider_specific_fields": message.get("provider_specific_fields"),
       "usage": body.get("usage"),
       "timings": body.get("timings"),
   })

   display(Markdown("### Qwen visible output\n\n```text\n" + (visible_text or "<EMPTY TEXT>") + "\n```"))
   print("=== QWEN VISIBLE OUTPUT ===", flush=True)
   print(visible_text or "<EMPTY TEXT>", flush=True)
   print("=== END QWEN VISIBLE OUTPUT ===", flush=True)
   return visible_text


def show_qwen_response(response, label: str = "qwen response") -> str:
   return show_qwen_body(response.model_dump(), label=label)


In [ ]:
# 4) Known-working raw HTTP Qwen call from src/minimal_cmbagent/backends.py
# Key detail: chat_template_kwargs={"enable_thinking": False}
payload = {
   "model": MODEL,
   "messages": [
       {
           "role": "system",
           "content": "You are a practical scientific coding assistant. Answer with final text only.",
       },
       {
           "role": "user",
           "content": "Write exactly: Qwen endpoint reachable",
       },
   ],
   "temperature": 0,
   "max_tokens": 200,
   "chat_template_kwargs": {"enable_thinking": False},
}

print("Sending raw HTTP request with enable_thinking=False...", flush=True)

with httpx.Client(timeout=180.0) as h:
   raw_response = h.post(
       f"{BASE_URL.rstrip('/')}/chat/completions",
       headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
       json=payload,
   )

print("Raw HTTP status:", raw_response.status_code, flush=True)
raw_response.raise_for_status()
raw_body = raw_response.json()
raw_qwen_text = show_qwen_body(raw_body, label="4 / raw HTTP qwen no-thinking call")


In [ ]:
# 5) Same no-thinking Qwen call through the OpenAI client
print("Sending OpenAI-client request with enable_thinking=False...", flush=True)

response = client.chat.completions.create(
   model=MODEL,
   messages=[
       {
           "role": "system",
           "content": "You are a practical scientific coding assistant. Answer with final text only.",
       },
       {
           "role": "user",
           "content": "Write exactly: Qwen endpoint reachable",
       },
   ],
   temperature=0,
   max_tokens=200,
   extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

qwen_text = show_qwen_response(response, label="5 / openai client qwen no-thinking call")


In [ ]:
# 6) Streaming no-thinking test
print("Streaming Qwen with enable_thinking=False...", flush=True)

stream = client.chat.completions.create(
   model=MODEL,
   messages=[{"role": "user", "content": "Say hello in one short sentence."}],
   temperature=0.2,
   max_tokens=200,
   stream=True,
   extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

stream_parts = []
for chunk in stream:
   delta = chunk.choices[0].delta
   data = delta.model_dump() if hasattr(delta, "model_dump") else {}
   text = (
       getattr(delta, "content", None)
       or getattr(delta, "reasoning_content", None)
       or data.get("reasoning_content")
       or data.get("provider_specific_fields", {}).get("reasoning_content")
       or ""
   )
   if text:
       stream_parts.append(text)
       print(text, end="", flush=True)

print("\n\n=== STREAM COMPLETE ===", flush=True)
stream_text = strip_template_tokens("".join(stream_parts).strip())
display(Markdown("### Qwen streaming output\n\n```text\n" + (stream_text or "<EMPTY STREAM>") + "\n```"))
display({"stream_text_repr": repr(stream_text), "stream_text_length": len(stream_text)})


In [ ]:
# 7) Minimal agent using the same no-thinking Qwen client
import math


def calculate(expression: str) -> str:
   if not re.fullmatch(r"[0-9+\-*/(). %eE]+", expression.strip()):
       return "Error: expression contains unsupported characters."
   try:
       return str(eval(expression, {"__builtins__": {}}, {"math": math}))
   except Exception as exc:
       return f"Error: {exc}"


TOOLS = {"calculate": calculate}

SYSTEM_PROMPT = """
You are a minimal tool-using agent.

If you need the calculator, reply with exactly one JSON object like this:
{"tool": "calculate", "input": "2 + 2"}

If you do not need a tool, answer normally and briefly.
Available tool:
- calculate: evaluate basic arithmetic.
""".strip()


def qwen_chat(messages, temperature=0.2, max_tokens=500):
   result = client.chat.completions.create(
       model=MODEL,
       messages=messages,
       temperature=temperature,
       max_tokens=max_tokens,
       extra_body={"chat_template_kwargs": {"enable_thinking": False}},
   )
   return normalize_qwen_response(result)


def extract_json_object(text):
   text = text.strip()
   if text.startswith("{") and text.endswith("}"):
       return json.loads(text)

   start = text.find("{")
   end = text.rfind("}")
   if start != -1 and end != -1 and end > start:
       return json.loads(text[start:end + 1])

   raise ValueError("No JSON object found")


def run_minimal_agent(user_task: str, max_steps: int = 4):
   messages = [
       {"role": "system", "content": SYSTEM_PROMPT},
       {"role": "user", "content": user_task},
   ]

   for step in range(max_steps):
       assistant_text = qwen_chat(messages)
       print(f"\n--- assistant step {step + 1} ---", flush=True)
       print(assistant_text or "<EMPTY TEXT>", flush=True)

       try:
           tool_call = extract_json_object(assistant_text)
       except Exception:
           return assistant_text

       tool_name = tool_call.get("tool")
       tool_input = tool_call.get("input")

       if tool_name not in TOOLS:
           return f"Unknown tool requested: {tool_name}"

       tool_result = TOOLS[tool_name](str(tool_input))
       print("\n--- tool result ---", flush=True)
       print(tool_result, flush=True)

       messages.append({"role": "assistant", "content": assistant_text})
       messages.append({
           "role": "user",
           "content": f"Tool result for {tool_name}: {tool_result}\nNow give the final answer only.",
       })

   return "Stopped: max agent steps reached."


In [ ]:
# 8) Minimal agent test
agent_answer = run_minimal_agent("Use the calculator tool: what is 12345 * 6789?")

print("\nFINAL AGENT ANSWER:", flush=True)
print(agent_answer or "<EMPTY TEXT>", flush=True)
display(Markdown("### Minimal agent final answer\n\n```text\n" + (agent_answer or "<EMPTY TEXT>") + "\n```"))
